# 01c Backdoor explorer (interactive)
Choose a dataset and a class, press **random image**: see the sample before and after the backdoor trigger is added.

* **FedUP sample trigger**: one pixel at the bottom-right corner; the released code writes 255 into the *normalised* tensor (about 63-79 in pixel units, shown saturated white). Triggered samples are relabelled to class 0.
* **FuGuard patch**: a k x k white square (k = 4 in the paper), relabelled to class 9.
* **AG News**: the token `the` (id 1996) is written at the last position of the padded sequence.

In [ ]:
# find the repo root (works from notebooks/<group>/, from the repo root, on Kaggle and on Colab) and set up the folders
import os, sys, shutil
from pathlib import Path
KAGGLE_REPO = Path("/kaggle/input/datasets/sakhiurrahman/fedupds/FedUP_Reproduction")   # your Kaggle dataset (edit if you move it)
def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working/FedUP_Reproduction"), Path("/content/FedUP_Reproduction"), KAGGLE_REPO]:
        if (p / "fedup_repro").exists():
            return p
    kin = Path("/kaggle/input")
    if kin.exists():                                    # any other layout of the input dataset, up to 6 levels deep
        for dp, dn, _ in os.walk(kin):
            if "fedup_repro" in dn:
                return Path(dp)
            if len(Path(dp).relative_to(kin).parts) > 6:
                dn[:] = []
    return None
ROOT = _find_repo()
assert ROOT is not None, "FedUP_Reproduction folder not found: upload it (Kaggle: as a Dataset) or cd into it"
if str(ROOT).startswith("/kaggle/input"):              # input datasets are read-only: work on a copy of the code ...
    SRC, ROOT = ROOT, Path("/kaggle/working") / ROOT.name
    if not ROOT.exists():
        shutil.copytree(SRC, ROOT, ignore=lambda d, names: [n for n in names if n == "raw" and Path(d).name == "data"])
    os.environ["FEDUP_BUNDLED_RAW"] = str(SRC / "data" / "raw")      # ... and LINK (not copy) the datasets that ship with it
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from fedup_repro import colab
PERSIST_ROOT = None        # Colab + Drive: "/content/drive/MyDrive/fedup_store" keeps data/ and artifacts/ across sessions (mount Drive first)
print("platform:", colab.setup_dirs(PERSIST_ROOT), "| repo:", ROOT)

In [ ]:
from fedup_repro import viz
viz.backdoor_explorer()

The same view without widgets (one figure, reproducible with `seed`):

In [ ]:
viz.show_backdoor('CIFAR-10', class_id=3, seed=1)
viz.show_backdoor('CIFAR-10', class_id=3, seed=1, trigger=list(viz.IMAGE_TRIGGERS)[2], patch=4)